# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

In [2]:
# constants

MODEL_GROQ = 'llama-3.3-70b-versatile'
MODEL_OLLAMA = 'llama3.2'

GROQ_BASE_URL = "https://api.groq.com/openai/v1"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [3]:
# set up environment

load_dotenv(override=True)

groq_client = OpenAI(
    base_url=GROQ_BASE_URL,
    api_key=os.getenv("GROQ_API_KEY")
)

ollama_client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"
)

system_prompt = """You are a helpful technical assistant that explains code and technical concepts clearly and concisely.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown."""

In [4]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""

In [5]:
# Get llama-3.3-70b via Groq to answer, with streaming

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

stream = groq_client.chat.completions.create(
    model=MODEL_GROQ,
    messages=messages,
    stream=True
)

for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="", flush=True)

### Explanation of the Code

The given code is a generator expression in Python that extracts authors from a list of books. 

* `books` is expected to be a list of dictionaries, where each dictionary represents a book.
* `book.get("author")` attempts to get the value associated with the key `"author"` from each book dictionary. If the key is not present, it returns `None` instead of raising an error.
* The `if book.get("author")` condition filters out books without an author, as `None` is considered falsey in Python.
* The `{...}` syntax is a set comprehension, which creates a set of unique authors. Sets in Python are unordered collections of unique elements.
* `yield from` is used to yield each item from the set of authors. This is equivalent to iterating over the set and yielding each item individually.

### Purpose of the Code

The purpose of this code is likely to extract a set of unique authors from a list of books, while ignoring books without an author. This can be useful in var

In [6]:
# Get Llama 3.2 via Ollama to answer (runs locally on your PC)

response = ollama_client.chat.completions.create(
    model=MODEL_OLLAMA,
    messages=messages
)

display(Markdown(response.choices[0].message.content))

**yield from Generator Expression**

The given code uses a generator expression to iterate over a list of `books` and extract the values of the `author` key from each book.

### Breakdown

* `yield from`: This keyword is used to delegate the execution of the corresponding iterable(s) to the yielded value.
* `{book.get("author") for book in books if book.get("author")}`: This is a generator expression that:
	1. Iterates over each element (`book`) in the list `books`.
	2. For each element, it tries to get the value of the key `"author"` using the `get()` method (assuming `book` is an object or dictionary-like). If the key does not exist, `None` will be returned.
	3. The expression returns this value (`author`) wrapped in a generator function.

### What happens when you execute this code?

1. When `yield from ...` encounters an iteration, it yields control to each item in the iterable (e.g., each book).
2. Inside the iteration:
	* If a book has an `"author"` key, its value is yielded.
	* If a book does not have an `"author"` key (or its value is `None`), this value (or `None`) is also yielded.

### Why use generator expressions?

Generator expressions are useful when:

* You need to process large datasets, but don't want to load the entire dataset into memory.
* You want to lazily compute values instead of immediately calculating them all at once.
* You're working with iterables and want a concise way to extract values from them.

In this case, the generator expression returns an iterator that yields book authors one by one, allowing you to process them individually without loading the entire list into memory.